# CLOY Processing

This notebook is for you to work with the CLOY data folder.

Turn every CLOY thread (a nested structure of the main post and subsequent comments) into a vector of words (as shown in slides 23-24 of Day 11 notes). [Spend some time looking at the structure of a single thread, since it is nested!]

Calculate the similarity score between every two threads  (with cosine similarity) [See sample code below], and save the dataframe into a CSV file.

Write a function that given one thread ID, prints out the threadID and subject title of the thread for the top 5 most similar threads.

Verification: Use the thread_explorer app to verify that these threads are indeed similar (you can search for the threads and read their content)

Do you think stop words are affecting your results?  What happens if you get rid of them? [TextBlob has a list of stop words.]

In [1]:
#setup
import pandas as pd

### 1) load in the cloy data

In [2]:
df = pd.read_json("data/RedditExplorer/cloy_chronological.jsonl", lines=True)
df.head()

,post_id,timestamp,author,subreddit,title,selftext,url,score,permalink,comments
0,dkxewd,2019-10-21 08:27:12,[deleted],CrashLandingOnYou,Crash Landing On You has been created,[deleted],https://www.reddit.com/r/CrashLandingOnYou/com...,1,https://reddit.com/r/CrashLandingOnYou/comment...,[]
1,ffh52z,2020-03-08 18:59:19,bruisedlee123,CrashLandingOnYou,"ok, let’s get this started! What are your thou...",I haven’t seen a kdrama for the longest time a...,https://www.reddit.com/r/CrashLandingOnYou/com...,28,https://reddit.com/r/CrashLandingOnYou/comment...,"[{'comment_id': 'fk0rooq', 'parent_id': 't3_ff..."
2,fguorr,2020-03-11 09:55:17,bruisedlee123,CrashLandingOnYou,Find someone who’ll make you noodles just like...,,https://i.redd.it/o8vnfixdn0m41.jpg,60,https://reddit.com/r/CrashLandingOnYou/comment...,"[{'comment_id': 'fm1mpbz', 'parent_id': 't3_fg..."
3,fhcrpl,2020-03-12 07:47:30,bruisedlee123,CrashLandingOnYou,Captain Ri’s squad! Who’s your favorite?,,https://i.redd.it/9a9p6mxh57m41.jpg,59,https://reddit.com/r/CrashLandingOnYou/comment...,"[{'comment_id': 'fkmslkl', 'parent_id': 't3_fh..."
4,fj5byj,2020-03-15 17:53:57,titania73,CrashLandingOnYou,Why I think we all need a second season of CLO...,I binged crash landing on you this week and ca...,https://www.reddit.com/r/CrashLandingOnYou/com...,20,https://reddit.com/r/CrashLandingOnYou/comment...,"[{'comment_id': 'fkkzuim', 'parent_id': 't3_fj..."


In [3]:
df.iloc[3]

post_id                                                 fhcrpl
timestamp                                  2020-03-12 07:47:30
author                                           bruisedlee123
subreddit                                    CrashLandingOnYou
title                 Captain Ri’s squad! Who’s your favorite?
selftext                                                      
url                        https://i.redd.it/9a9p6mxh57m41.jpg
score                                                       59
permalink    https://reddit.com/r/CrashLandingOnYou/comment...
comments     [{'comment_id': 'fkmslkl', 'parent_id': 't3_fh...
Name: 3, dtype: object

### 1) Understand nested structure

In [4]:
df.iloc[3]["comments"]

[{'comment_id': 'fkmslkl',
  'parent_id': 't3_fhcrpl',
  'author': 'SpottedFaun',
  'comment_body': "I can't narrow it down to just one. I love them all for completely different reasons. 😍",
  'score': 13,
  'created_utc': 1584338490},
 {'comment_id': 'fkryoce',
  'parent_id': 't3_fhcrpl',
  'author': 'Sierra2323',
  'comment_body': 'I love them all!!! 💗💗',
  'score': 11,
  'created_utc': 1584478674},
 {'comment_id': 'fll2r59',
  'parent_id': 't3_fhcrpl',
  'author': 'usernameisinvaliderr',
  'comment_body': 'I can’t choose either. Must be a sign of good casting/script/acting. I love them all!',
  'score': 9,
  'created_utc': 1585252529},
 {'comment_id': 'flxev91',
  'parent_id': 't3_fhcrpl',
  'author': 'genevievesprings',
  'comment_body': 'oml, they give me so much life &lt;3',
  'score': 4,
  'created_utc': 1585581495},
 {'comment_id': 'fm3gptu',
  'parent_id': 't3_fhcrpl',
  'author': 'Cerenas',
  'comment_body': "The main reason I like this show. Their scenes are funny, especiall

In [5]:
df.iloc[3]["comments"][4]["replies"]

[{'comment_id': 'fm3hjy8',
  'parent_id': 't1_fm3gptu',
  'author': 'bruisedlee123',
  'comment_body': 'Oh cool! A dude who likes CLOY. Anyways their scenes at the jimjilbang/Korean spa were funny. And their characters were consistent! Like Pyo Chi Su getting the face scrub...he’s the metrosexual of the group 😆',
  'score': 5,
  'created_utc': 1585723754,
  'replies': [{'comment_id': 'fm3j8ao',
    'parent_id': 't1_fm3hjy8',
    'author': 'Cerenas',
    'comment_body': 'And the love/hate relationship between Se-Ri and Pyo Chi Su is funny as well.\n\nReally glad that stuff was in there, because boy what was this serie emotionally draining. Finished it last night and now I have to live with the void that it left.',
    'score': 10,
    'created_utc': 1585725423,
    'replies': [{'comment_id': 'fm3jtba',
      'parent_id': 't1_fm3j8ao',
      'author': 'bruisedlee123',
      'comment_body': 'Sigh. I can relate. I still can’t move on to watching another kdrama. Hang in there! 😔☺️',
      '

### 1) Thread into vector of words

In [6]:
#do thread just for 1 item
thread = df.iloc[3]
texts = []
#add the selftext and title
texts.append(thread["selftext"])
texts.append(thread["title"])

#get text from the replies
def get_text(comment):
    texts.append(comment["comment_body"])
    #get text from each reply and skip if it doesnt exist
    for reply in comment.get("replies", []):
        get_text(reply)

#get text in every comment
for comment in thread["comments"]:
    get_text(comment)

#join all the text chunks into 1 big text
all_text = " ".join(texts)

#make big text into a word vector
words = all_text.split()

words

['Captain',
 'Ri’s',
 'squad!',
 'Who’s',
 'your',
 'favorite?',
 'I',
 "can't",
 'narrow',
 'it',
 'down',
 'to',
 'just',
 'one.',
 'I',
 'love',
 'them',
 'all',
 'for',
 'completely',
 'different',
 'reasons.',
 '😍',
 'I',
 'love',
 'them',
 'all!!!',
 '💗💗',
 'I',
 'can’t',
 'choose',
 'either.',
 'Must',
 'be',
 'a',
 'sign',
 'of',
 'good',
 'casting/script/acting.',
 'I',
 'love',
 'them',
 'all!',
 'oml,',
 'they',
 'give',
 'me',
 'so',
 'much',
 'life',
 '&lt;3',
 'The',
 'main',
 'reason',
 'I',
 'like',
 'this',
 'show.',
 'Their',
 'scenes',
 'are',
 'funny,',
 'especially',
 'when',
 "they're",
 'in',
 'the',
 'South.',
 'My',
 'gf',
 'got',
 'me',
 'watching',
 'along',
 'and',
 "I'm",
 'not',
 'that',
 'into',
 'drama,',
 'so',
 'I',
 'was',
 'really',
 'happy',
 'this',
 'silly',
 'group',
 'was',
 'there.',
 'Oh',
 'cool!',
 'A',
 'dude',
 'who',
 'likes',
 'CLOY.',
 'Anyways',
 'their',
 'scenes',
 'at',
 'the',
 'jimjilbang/Korean',
 'spa',
 'were',
 'funny.',
 'And

In [7]:
def get_text(comment, texts):
    texts.append(comment["comment_body"])

    for reply in comment.get("replies", []):
        get_text(reply, texts)

all_threads = []

for index, thread in df.iterrows():
    texts = []

    texts.append(thread["title"])
    texts.append(thread["selftext"])

    for comment in thread["comments"]:
        get_text(comment, texts)

    all_text = " ".join(texts)
    words = all_text.split()

    all_threads.append(words)

all_threads

[['Crash', 'Landing', 'On', 'You', 'has', 'been', 'created', '[deleted]'],
 ['ok,',
  'let’s',
  'get',
  'this',
  'started!',
  'What',
  'are',
  'your',
  'thoughts',
  'about',
  'the',
  'show?',
  'I',
  'haven’t',
  'seen',
  'a',
  'kdrama',
  'for',
  'the',
  'longest',
  'time',
  'and',
  'it',
  'was',
  'actually',
  'my',
  'dad',
  'who',
  'chose',
  'to',
  'stream',
  'CLOY',
  'on',
  'Netflix.',
  'He',
  'stopped',
  'watching',
  'after',
  'the',
  'second',
  'episode',
  'but',
  'I',
  'went',
  'on',
  'after',
  'immediately',
  'falling',
  'in',
  'love',
  'with',
  'the',
  'leads.',
  'Hyun',
  'Bin',
  'is',
  'just',
  'so',
  'charismatic',
  'as',
  'Capt.',
  'Ri',
  'Jeong',
  'Hyeok',
  'and',
  'Son',
  'Ye-jin',
  'is',
  'perfect',
  'as',
  'Yoon',
  'Se-ri.',
  'She’s',
  'definitely',
  'got',
  'the',
  'rom-com',
  'and',
  'drama',
  'genre',
  'down',
  'pat!',
  'I',
  'also',
  'loved',
  'Capt.',
  'Ri’s',
  'squad,',
  'most',
  '

### 2) Calculate cosine similarity

In [8]:
#calculate cosine similarity between every two threads

from textblob import TextBlob
from collections import Counter
import numpy as np

# thread id
ids = df["post_id"]

#clean up text of thread into all lowercase + remove punchuation 
#make into string and then .words for list
clean_docs = [TextBlob(" ".join(words).lower()).words for words in all_threads]

In [9]:
#build a unique word table to count how many times a certain word appears in a thread
def make_matrix(stop=set()):
    docs = [[w for w in doc if w not in stop] for doc in clean_docs]
    return pd.DataFrame([Counter(d) for d in docs], index=ids).fillna(0)

### 2) save df into csv file of similarities

In [10]:
#return cosine similarity table between word tables
def cosine(M):
    A = M.values.astype("float32")
    norms = np.sqrt((A ** 2).sum(axis=1))
    norms[norms == 0] = 1
    similaritytable = (A @ A.T) / np.outer(norms, norms)
    return pd.DataFrame(similaritytable, index=M.index, columns=M.index)

similaritytable_df = cosine(make_matrix())
similaritytable_df.to_csv("thread_similarity.csv")

In [11]:
print(similaritytable_df.shape)
print(similaritytable_df.iloc[0, 0])

(3075, 3075)
1.0000001


### 3) given 1 thread_id print out thread_id and title for top 5 most similar threads

In [12]:
#get top 5 most similar threads to the selected thread
def top5(thread_id, similaritytable=similaritytable_df):
    top = similaritytable[thread_id].drop(thread_id).sort_values(ascending=False).head(5)
    for topid in top.index:
        print(topid, "|", df.loc[df["post_id"] == topid, "title"].iloc[0])

top5(ids.iloc[0])

zoafq7 | [FULL LYRIC] CRASH LANDING ON YOU
g6oy82 | My Crash Landing on You Penguin art!! :D
1ntfysd | CRASH LANDING ON YOU PLOT HOLES
fqh4pi | Crash Landing on You Season 2 Details!
1ntfyai | Crash Landing on You weird plot holes


In [ ]:
from nltk.corpus import stopwords 
import nltk

stop = set(stopwords.words("english"))

similaritytable_df2 = cosine(make_matrix(stop))

print("------------original thread")
ids.iloc[0]
print("------------with stop words")
top5(ids.iloc[0])
print("------------without stop words")
top5(ids.iloc[0], similaritytable=similaritytable_df2)

------------with stop words
zoafq7 | [FULL LYRIC] CRASH LANDING ON YOU
g6oy82 | My Crash Landing on You Penguin art!! :D
1ntfysd | CRASH LANDING ON YOU PLOT HOLES
fqh4pi | Crash Landing on You Season 2 Details!
1ntfyai | Crash Landing on You weird plot holes
------------without stop words
g6oy82 | My Crash Landing on You Penguin art!! :D
zoafq7 | [FULL LYRIC] CRASH LANDING ON YOU
wrejnf | photos from the below site:Information on items and fashions of crash landing on you
k3bb2r | Crash Landing On You - OST 'All of My Days' - translated by native Korean
jptxvi | What did Seri mean?


Do you think stop words are affecting your results?  What happens if you get rid of them? [TextBlob has a list of stop words.]

I think stop words are affecting my results because when i remove the stop words, my top 5 threads changed significantly. The original ones are shorter which help the cosine similarity but after removing top words, it seems like maybe the top topics relating to the first thread is better. 